In [ ]:
import os
import requests
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from dotenv import load_dotenv

# Load API key
load_dotenv()
API_KEY = os.getenv("API_KEY")

# Input: parquet of main games table
GAMES_PARQUET_DIR = "data/games"
OUTPUT_PARQUET_DIR = "data/games_details"

# Ensure output folder exists
os.makedirs(OUTPUT_PARQUET_DIR, exist_ok=True)

def fetch_game_details(game_id):
    """
    Call GameBrain API for a single game's details
    """
    url = f"https://api.gamebrain.co/v1/games/{game_id}"
    headers = {"x-api-key": API_KEY}
    resp = requests.get(url, headers=headers)
    
    if resp.status_code != 200:
        print(f"Failed to fetch {game_id}: {resp.status_code}")
        return None
    
    return resp.json()


def transform_game_details(game_json):
    """
    Flatten game JSON into a single row for parquet
    """
    if not game_json:
        return None
    
    row = {
        "game_id": game_json.get("id"),
        "name": game_json.get("name"),
        "release_date": game_json.get("release_date"),
        "developer": game_json.get("developer"),
        "genre": game_json.get("genre"),
        "adult_only": game_json.get("adult_only"),
        # first screenshot / video only
        "screenshot": game_json.get("screenshots")[0] if game_json.get("screenshots") else None,
        "video": game_json.get("videos")[0] if game_json.get("videos") else None,
        "micro_trailer": game_json.get("micro_trailer"),
        # first offer only
        "offer_price": game_json.get("offers")[0]["price"].get("value") if game_json.get("offers") else None,
        "offer_currency": game_json.get("offers")[0]["price"].get("currency") if game_json.get("offers") else None,
        "offer_discount_percent": game_json.get("offers")[0]["price"].get("discount_percent") if game_json.get("offers") else None,
        "offer_store": game_json.get("offers")[0].get("store_name") if game_json.get("offers") else None,
        "offer_platform": game_json.get("offers")[0].get("platform") if game_json.get("offers") else None,
        "official_store_1_source": game_json.get("official_stores")[0]["source"] if game_json.get("official_stores") else None,
        "official_store_1_url": game_json.get("official_stores")[0]["url"] if game_json.get("official_stores") else None,
        "official_store_2_source": game_json.get("official_stores")[1]["source"] if len(game_json.get("official_stores", [])) > 1 else None,
        "official_store_2_url": game_json.get("official_stores")[1]["url"] if len(game_json.get("official_stores", [])) > 1 else None,
    }
    return row


def process_games_parquet(input_dir, output_dir):
    """
    Iterate all parquet files in the games folder
    """
    all_files = []
    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.endswith(".parquet"):
                all_files.append(os.path.join(root, file))
    
    for parquet_file in all_files:
        print(f"Processing {parquet_file}")
        df = pd.read_parquet(parquet_file)
        
        # Iterate game_ids
        details_rows = []
        for game_id in df['game_id']:
            game_json = fetch_game_details(game_id)
            row = transform_game_details(game_json)
            if row:
                details_rows.append(row)
        
        if details_rows:
            df_details = pd.DataFrame(details_rows)
            # Build partitioning
            year_partition = df['year_partition'].iloc[0]
            month_partition = df['month_partition'].iloc[0]
            output_path = os.path.join(output_dir, f"year={year_partition}", f"month={month_partition:02d}")
            os.makedirs(output_path, exist_ok=True)
            file_path = os.path.join(output_path, f"games_details_{year_partition}_{month_partition:02d}.parquet")
            
            table = pa.Table.from_pandas(df_details)
            pq.write_table(table, file_path)
            print(f"Saved details: {file_path}")


if __name__ == "__main__":
    process_games_parquet(GAMES_PARQUET_DIR, OUTPUT_PARQUET_DIR)

Processing data/games\year=2025\month=10\games_2025_10.parquet


KeyError: 'discount_percent'